# Notebook 02 of 7 — Single-Name Deep Dive

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

In NB01 I met the plumbing. Now I'm going to put one name through it end-to-end. I picked MSFT — it's the largest single position in my basket, and 'is MSFT a good company' is exactly the kind of question I used to answer with a gut feel. Let's replace that.

By the end of this notebook we will be able to answer one question:

> *What do I actually think of MSFT — as a repeatable, phase-by-phase answer instead of a hot take?*


## 0. Why start with one name

There's a discipline I stole from a much better trader: **never analyze
a basket before you can analyze its members**. Portfolio-level
concentration numbers are meaningless if you don't have a per-name
opinion to concentrate on.

So this notebook builds the per-name opinion. The next one
(NB03) uses it.

*The code cell below runs the full 7-phase pipeline for MSFT once,
then we walk each phase individually below.*

### Fundamental vs technical — and why the pipeline uses both

Every guide splits stock analysis into two camps and then tells you which one is right. Both camps are useful; neither is sufficient. Here's the mental model I use across the 7 phases:

> **📖 Fundamental analysis** — evaluating a stock by the underlying business: earnings, growth, balance-sheet health, competitive position, cash flow, management. Phases 1, 2, 4, and 6 of the pipeline are fundamental. Answers "is this a good company at a fair price?" [Investopedia on fundamental analysis →](https://www.investopedia.com/terms/f/fundamentalanalysis.asp)

> **📖 Technical analysis** — evaluating a stock by the price + volume history alone: trend, momentum, support/resistance, chart patterns. Phase 3 is technical. Answers "what is the tape saying right now?" [Investopedia on technical analysis →](https://www.investopedia.com/terms/t/technicalanalysis.asp)

The pipeline uses both, in that order: fundamentals form the thesis, technicals form the *sanity check*. A great business trading at a fair valuation whose price has been in a six-month waterfall on rising volume is a signal that someone with better information than me has been selling. That's not a veto — it's an input to Phase 7's composite. The whole point of a composite score is that no single phase gets to unilaterally decide.

### Provider chain for this notebook (Track A / #1430)

NB02's pipeline runs exclusively on `fmp_cached` (my paid + cached tier).
That's the primary. But every fundamentals / price call in the platform
falls back through the same authoritative chain the series adopted in
[NB01 §2](./01-getting-started-and-providers.ipynb):

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

Concretely for this notebook:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| Company profile + fundamentals | `fmp_cached` | SEC XBRL **company facts** (`sec`) |
| Prices / OHLCV | `fmp_cached` | `cboe` (EOD) |
| Analyst consensus / price target | `fmp_cached` | *no free authoritative source — gap; keep on `fmp`* |
| Everything else | `fmp_cached` | `sec` → `cboe` → yfinance (labeled) |

> **📖 SEC XBRL is as-filed.** Fundamentals from SEC EDGAR reflect what
> the company filed on the day it filed it. Restatements appear as
> *later* filings — the earlier value stays visible. That is a feature
> for point-in-time backtesting and a caveat for anyone reading a
> single row: "as of date X" ≠ "the current company view of date X".

No code cells below change under this PR — the pipeline is already
authoritative. The note above documents the chain so a reader knows
which fallback is next when a call raises.


In [ ]:
# [Phase B / NB02 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


In [ ]:
# [Phase B / NB02 §0] one shot — run all 7 phases on MSFT, walk them below
# Uses fmp_cached as the primary provider (see Analysis/stock_analysis.py
# PRIMARY_PROVIDER = "fmp_cached"). Typical wall-clock on a warm cache:
# 20-25 seconds. Cold cache: significantly slower (rerun to warm).
import sys, time
sys.path.insert(0, "../../Analysis")  # notebook CWD is notebooks/portfolio/
from stock_analysis import AnalysisConfig, run_full_analysis

t0 = time.perf_counter()
result = run_full_analysis(AnalysisConfig(symbol="MSFT"))
dt = time.perf_counter() - t0

print(f"7 phases in {dt:.1f}s   phase keys: {sorted(result.keys())}")
for k in sorted(result.keys()):
    v = result.get(k)
    print(f"  {k}: {type(v).__name__ if v is not None else 'None'}")


Dropping institutional-ownership record for MSFT due to schema mismatch: 3 validation errors for FMPInstitutionalOwnershipData
ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
last_ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
ownership_percent_change
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


...

Failed to fetch dividends for DOCN: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPSC: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for XLK: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


7 phases in 9.3s   phase keys: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']
  p1: Phase1Result
  p2: Phase2Result
  p3: Phase3Result
  p4: Phase4Result
  p5: Phase5Result
  p6: Phase6Result
  p7: Phase7Result


## 1. Phase 1 — Company

**Question:** *Who are they, what do they sell, how is the share structure?*

This is the boring phase and the most important one. If I can't explain in one paragraph what MSFT sells and to whom, I have no business owning it. Phase 1 forces me to.

*The code cell below renders phase 1's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 Shares outstanding** — total shares the company has issued and that are held by all shareholders (including insiders and restricted holders). Multiply by price for market cap. [Investopedia on shares outstanding →](https://www.investopedia.com/terms/o/outstandingshares.asp)

> **📖 Institutional ownership** — the fraction of shares held by mutual funds, pension funds, ETFs, and other institutions (reported quarterly via 13F filings, which NB04 covers in depth). Very high (>90%) means the price is largely set by big-money flows; very low (<20%) means retail sentiment can dominate. MSFT sits north of 70%. [Investopedia on institutional ownership →](https://www.investopedia.com/terms/i/institutionalownership.asp)

> **📖 Peer group** — the small set of companies you compare against because they operate in the same business, of roughly similar size, with roughly similar economics. Phase 1 picks the peer set; Phase 6 uses it to rank MSFT's metrics. Bad peer set = bad relative-value verdict. [Investopedia on peer group →](https://www.investopedia.com/terms/p/peer-group.asp)

In [ ]:
# [Phase B / NB02 §1] Phase 1 — Company (who they are, share structure, insider/institutional makeup)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p1"]
if p is None:
    print(f"Phase 1: no result (upstream failure)")
else:
    print(f"Phase 1 — Company (who they are, share structure, insider/institutional makeup)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 1 — Company (who they are, share structure, insider/institutional makeup)
  gate_passed:  True
  earnings_revision_3m_direction:  down
  free_float_pct:                  None
  geo_df:                          DataFrame  shape=(26, 5)
  industry:                        
  insider_df:                      DataFrame  shape=(20, 16)
  institutional_df:                DataFrame  shape=(0, 0)
  market_cap:                      2835431731000.0
  metrics_df:                      DataFrame  shape=(1, 47)
  peers:                           list  len=9  first=NVDA
  price_targets_df:                DataFrame  shape=(100, 9)
  profile_df:                      DataFrame  shape=(1, 32)
  quote_df:                        DataFrame  shape=(1, 17)
  sector:                          Technology
  short_interest_pct:              None
  gate_notes (2):
    - O
    - K


## 2. Phase 2 — Fundamentals

**Question:** *Are they earning real money, is it growing, is the balance sheet clean?*

The three-statement basics — plus the *ratios that catch nonsense*: owner earnings vs reported net income, cash-conversion, working-capital trend. If those look ugly, nothing later in this pipeline saves the pick.

*The code cell below renders phase 2's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

### Fundamentals — the four numbers that survive most audits

Phase 2 pulls dozens of ratios and hands them to Phase 7 as a health-score. Four of them do most of the load-bearing work:

> **📖 Owner earnings** — Warren Buffett's alternative to reported net income: net income + depreciation & amortization − maintenance capex − working-capital changes. Aims to be "the cash a shareholder could actually take out of the business without impairing it." Reported EPS can be gamed by accounting choices; owner earnings is a lot harder to inflate. [Investopedia on owner's earnings →](https://www.investopedia.com/terms/o/ownersearnings.asp)

> **📖 ROIC (return on invested capital)** — after-tax operating profit divided by the capital (equity + interest-bearing debt) tied up in the business. Answers: for every dollar the business has consumed, how many cents does it produce annually? Above the company's cost of capital = creating value; below = destroying value. [Investopedia on ROIC →](https://www.investopedia.com/terms/r/returnoninvestmentcapital.asp)

> **📖 ROE (return on equity)** — net income divided by shareholder equity. Simpler and older than ROIC; noisier because it can be inflated by leverage. Read ROE and ROIC together — a widening gap usually means the company is borrowing to buy back stock. [Investopedia on ROE →](https://www.investopedia.com/terms/r/returnonequity.asp)

> **📖 Piotroski F-score** — a 9-point checklist of accounting-quality signals (profitability, leverage, operating efficiency). 8 or 9 = clean books, low probability of a shock. 0-2 = something is off. Cheap and blunt; catches distress that individual ratios miss. [Investopedia on the Piotroski F-score →](https://www.investopedia.com/terms/p/piotroski-score.asp)

> **📖 Altman Z-score** — a distress-prediction score, originally calibrated on manufacturers. Above 3 = safe zone; below 1.8 = distress zone; the middle is grey. Not perfect on modern asset-light businesses (software, services) but still a useful "is bankruptcy remotely on the table?" gate. [Investopedia on the Altman Z-score →](https://www.investopedia.com/terms/a/altman.asp)

In [ ]:
# [Phase B / NB02 §2] Phase 2 — Fundamentals (earnings quality, balance-sheet health)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p2"]
if p is None:
    print(f"Phase 2: no result (upstream failure)")
else:
    print(f"Phase 2 — Fundamentals (earnings quality, balance-sheet health)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 2 — Fundamentals (earnings quality, balance-sheet health)
  gate_passed:  False
  accruals_ratio:                  nan
  balance_df:                      DataFrame  shape=(5, 61)
  cash_df:                         DataFrame  shape=(5, 47)
  dilution_5y:                     0.019156061620897447
  gross_profitability:             0.3471039220562108
  income_df:                       DataFrame  shape=(5, 39)
  kpi_df:                          DataFrame  shape=(1, 20)
  operating_leverage:              nan
  ratios_df:                       DataFrame  shape=(1, 63)
  roe_decomp_df:                   DataFrame  shape=(0, 0)
  score:                           3.446
  gate_notes (21):
    - S
    - c
    - o
    - r
    - e


## 3. Phase 3 — Technicals

**Question:** *What is the tape saying — trend, momentum, volume?*

I don't trade on technicals. I *sanity-check* on technicals. If fundamentals say 'great, buy' and price has been in a 6-month waterfall on rising volume, someone knows something I don't.

*The code cell below renders phase 3's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 ATR (average true range)** — a rolling 14-day average of the daily true range (max of today's high-low, |today's high − yesterday's close|, |today's low − yesterday's close|). Reads out as a dollar amount of typical daily volatility. Phase 7's staged-entry protocol sizes the trailing stop in units of ATR so the stop scales with the name's actual volatility instead of an arbitrary percentage. [Investopedia on ATR →](https://www.investopedia.com/terms/a/atr.asp)

> **📖 Stop-loss order** — a resting order that sells the position if price falls to a level you pre-committed to. Turns "how much am I willing to lose on this?" from a heat-of-the-moment decision into a policy. [Investopedia on stop-loss →](https://www.investopedia.com/terms/s/stop-lossorder.asp)

> **📖 Trailing stop** — a stop-loss that ratchets up (never down) as price rises. Locks in gains without capping upside. Phase 7 emits both the initial stop and the trailing rule in one bundle for NB05 to translate into broker orders. [Investopedia on trailing stops →](https://www.investopedia.com/terms/t/trailingstop.asp)

In [ ]:
# [Phase B / NB02 §3] Phase 3 — Technicals (trend, momentum, volume; earnings-safe-window)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p3"]
if p is None:
    print(f"Phase 3: no result (upstream failure)")
else:
    print(f"Phase 3 — Technicals (trend, momentum, volume; earnings-safe-window)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 3 — Technicals (trend, momentum, volume; earnings-safe-window)
  gate_passed:  False
  atr:                             11.832048891098891
  bullish_count:                   5
  days_to_earnings:                1
  earnings_safe_window:            False
  entry_quality:                   Cautious
  extended_panel:                  None
  fib_levels:                      dict  keys=['23.6%', '38.2%', '50.0%', '61.8%', '78.6%']...
  price_df:                        DataFrame  shape=(312, 36)
  signals:                         dict  keys=['above_cloud', 'above_vwap', 'adx_trending', 'bb_squeeze_breakout', 'cmf_positive']...
  weekly_trend_bullish:            False
  gate_notes (62):
    - 5
    - /
    - 1
    - 3
    -  


## 4. Phase 4 — Valuation

**Question:** *Is the price sensible against intrinsic and relative anchors?*

Multi-anchor: a DCF I don't trust in isolation, a relative-multiple band vs. peer set, and an owner-earnings yield vs. the risk-free rate. If two of three agree the name is overpriced, that's the input to phase 7, not a veto by itself.

*The code cell below renders phase 4's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

### Multi-anchor valuation — why no single number wins

Phase 4 refuses to output a single "fair value." It outputs a *band* built from three independent anchors, because every single valuation method has a known failure mode and the failures don't overlap. Sam's rule: if two of three anchors agree the name is expensive, that's a real vote toward expensive.

> **📖 DCF (discounted cash flow)** — project the business's future free cash flows out ~10 years, discount each year back to today using a required-return rate, sum. Result = "what this business is worth if my forecast + discount rate are right." Extremely sensitive to the terminal-growth and discount-rate inputs — small tweaks swing the answer wildly. Never a standalone verdict. [Investopedia on DCF →](https://www.investopedia.com/terms/d/dcf.asp)

> **📖 Enterprise value (EV)** — market cap + total debt − cash & equivalents. What you'd have to pay to buy the whole business free of its capital structure. Used as the numerator in cross-company valuation multiples because it's structure-neutral. [Investopedia on enterprise value →](https://www.investopedia.com/terms/e/enterprisevalue.asp)

> **📖 EV / EBITDA** — enterprise value divided by earnings before interest, taxes, depreciation, and amortization. The go-to "how expensive is this business's operating engine?" multiple. Comparable across capital structures in a way P/E is not. [Investopedia on EV/EBITDA →](https://www.investopedia.com/terms/e/ev-ebitda.asp)

> **📖 EV / Sales** — enterprise value divided by revenue. Blunter than EV/EBITDA; the fallback when a business is unprofitable or when EBITDA is being aggressively adjusted. Used to sanity-check high-growth names. [Investopedia on EV/Sales →](https://www.investopedia.com/terms/e/enterprisevaluesales.asp)

> **📖 Price target + analyst rating** — sell-side analysts publish a 12-month price target and a Buy/Hold/Sell rating; the *consensus* is the average across analysts covering the name. Useful as a sanity check on where the crowd is, not as a directional signal — the consensus is a lagging aggregate and the dispersion (max − min) often carries more information than the mean. [Investopedia on price targets →](https://www.investopedia.com/terms/p/pricetarget.asp) · [Investopedia on analyst ratings →](https://www.investopedia.com/terms/a/analystratings.asp)

In [ ]:
# [Phase B / NB02 §4] Phase 4 — Valuation (DCF + multi-anchor; entry recommendation)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p4"]
if p is None:
    print(f"Phase 4: no result (upstream failure)")
else:
    print(f"Phase 4 — Valuation (DCF + multi-anchor; entry recommendation)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 4 — Valuation (DCF + multi-anchor; entry recommendation)
  gate_passed:  False
  altman:                          nan
  dcf_fair_value:                  129.49810968141492
  entry_recommendation:            Avoid — overvalued regardless of technicals
  historical_multiples_df:         DataFrame  shape=(1, 4)
  implied_growth:                  nan
  margin_of_safety:                -1.9475333727962527
  multiples_df:                    DataFrame  shape=(1, 10)
  multiples_vs_median:             dict  keys=['ev_ebitda', 'p_fcf', 'p_s', 'pe']...
  peg_ratio:                       nan
  piotroski:                       nan
  roic_wacc_spread:                nan
  sensitivity_df:                  DataFrame  shape=(3, 3)
  valuation_verdict:               Overvalued
  gate_notes (85):
    - M
    - O
    - S
    -  
    - -


## 5. Phase 5 — Risk

**Question:** *What happens if I am wrong — drawdown, vol, correlation?*

Not risk in isolation — risk *relative to what I already own*. MSFT's beta only matters once I know NB03's basket-level exposure. This phase produces the numbers NB03 will merge in.

*The code cell below renders phase 5's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 Beta** — the slope of the regression of a stock's returns against a market index's returns. Beta = 1 means the stock moves 1-for-1 with the market on average; beta = 1.4 means 40% more amplitude; negative beta means it tends to move opposite. Not a promise about tomorrow — a description of the past correlation window. [Investopedia on beta →](https://www.investopedia.com/terms/b/beta.asp)

> **📖 Conditional Value at Risk (CVaR / expected shortfall)** — the average loss on the *worst* p% of historical days (e.g. CVaR-5% = mean loss on the worst 5% of days). Value-at-Risk gives you the threshold; CVaR gives you the average of the tail beyond the threshold, which is what actually happens when things go wrong. [Investopedia on CVaR →](https://www.investopedia.com/terms/c/conditional_value_at_risk.asp)

In [ ]:
# [Phase B / NB02 §5] Phase 5 — Risk (beta, drawdown, CVaR, conviction sizing)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p5"]
if p is None:
    print(f"Phase 5: no result (upstream failure)")
else:
    print(f"Phase 5 — Risk (beta, drawdown, CVaR, conviction sizing)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 5 — Risk (beta, drawdown, CVaR, conviction sizing)
  gate_passed:  True
  beta:                            0.8048962844762665
  beta_down:                       0.7521608072787765
  beta_up:                         0.6169122726826494
  calmar:                          0.04396414578711472
  conviction_size:                 0.01
  cvar_95:                         -0.037734983634824956
  gain_to_pain:                    1.010549743115057
  half_kelly_size:                 0.00010875990258320003
  kelly_fraction:                  0.005437995129160001
  kurtosis:                        5.497198733094857
  max_drawdown:                    -0.3491062039957952
  portfolio_fit:                   Satellite
  recommended_size:                0.00010875990258320003
  risk_kpi_df:                     DataFrame  shape=(1, 17)
  sharpe:                          -0.017544915747477633
  skewness:                        -0.2830272834341935
  sortino:                         -0.023623697056066525
 

## 6. Phase 6 — Peer-relative

**Question:** *How does MSFT rank against its peer set on every metric that mattered?*

Same metrics as phases 2, 4, 5 — but ranked against the peer set from phase 1. This is where 'MSFT looks cheap' either survives or dies.

*The code cell below renders phase 6's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

In [ ]:
# [Phase B / NB02 §6] Phase 6 — Peer-relative (rank vs peer set; momentum accel)
# Render a compact table of the most useful fields (not the full result
# object). Surface any warnings the phase emitted via gate_notes.
p = result["p6"]
if p is None:
    print(f"Phase 6: no result (upstream failure)")
else:
    print(f"Phase 6 — Peer-relative (rank vs peer set; momentum accel)")
    print(f"  gate_passed:  {getattr(p, 'gate_passed', 'n/a')}")
    # Print each non-DF, non-underscore field
    for name in sorted(dir(p)):
        if name.startswith("_") or name in ("gate_passed", "gate_notes"):
            continue
        try:
            val = getattr(p, name)
        except Exception:
            continue
        if callable(val):
            continue
        # Skip DataFrames + huge collections; only print scalars/short lists
        tname = type(val).__name__
        if tname == "DataFrame":
            print(f"  {name+':':<32} DataFrame  shape={val.shape}")
            continue
        if isinstance(val, dict):
            print(f"  {name+':':<32} dict  keys={sorted(val.keys())[:5]}...")
            continue
        if isinstance(val, (list, tuple)) and len(val) > 5:
            print(f"  {name+':':<32} {tname}  len={len(val)}  first={val[0]}")
            continue
        print(f"  {name+':':<32} {val}")
    notes = getattr(p, "gate_notes", None) or []
    if notes:
        print(f"  gate_notes ({len(notes)}):")
        for note in notes[:5]:
            print(f"    - {note}")


Phase 6 — Peer-relative (rank vs peer set; momentum accel)
  gate_passed:  False
  corr_matrix:                     DataFrame  shape=(11, 11)
  information_ratio:               -1.5363194002105482
  momentum_accel_63d:              -0.0909090909090909
  peer_fundamental_df:             DataFrame  shape=(8, 46)
  relative_score:                  2.371
  relative_table:                  DataFrame  shape=(11, 9)
  relative_valuation_score:        50.0
  rolling_3m_rank:                 18.181818181818183
  sector_etf:                      XLK
  gate_notes (94):
    - R
    - e
    - l
    - a
    - t


## 7. Phase 7 — Composite decision

**Question:** *One label, one entry-quality tag, one staged-entry protocol?*

Not a Buy/Hold/Sell hot take. A composite score with an entry-quality band (Strong / Fair / Poor / Avoid) and — critically — a staged-entry protocol: what size to open at, what triggers a second tranche, what the trailing-stop rule is. This is what actually gets handed to NB05 when we start placing orders.

*The code cell below renders phase 7's result as a short
table, and prints any warnings the phase emitted (thin peer set,
missing statement, regime hostility, etc.).*

> **📖 Composite score** — a single number formed by weighting several sub-scores according to a fixed policy. The value of a composite is *not* precision (any single sub-score is noisier); it's that the weighting is decided *before* seeing the data, so you cannot cherry-pick which sub-score to lean on after the fact. Phase 7's composite blends Phases 1-6 under a regime overlay and outputs an entry-quality band (Strong / Fair / Poor / Avoid) plus a staged-entry protocol.

In [ ]:
# [Phase B / NB02 §7] Phase 7 — Composite decision (label + entry-quality + handoff)
p7 = result["p7"]
if p7 is None:
    print("Phase 7: no result")
else:
    print(f"COMPOSITE DECISION")
    print(f"  action_label:     {p7.action_label}")
    print(f"  composite_score:  {getattr(p7, 'composite_score', 'n/a')}")
    print(f"  entry_quality:    {p7.entry_quality}")
    print(f"  regime (applied): {getattr(p7, 'regime', 'n/a')}")
    print(f"  hard_override:    {getattr(p7, 'hard_override', 'n/a')}")
    print(f"  atr_stop:         {getattr(p7, 'atr_stop', 'n/a')}")
    triggers = getattr(p7, 'monitoring_triggers', None)
    if triggers:
        # monitoring_triggers is a dict{trigger_name -> level}, not a list
        n = len(triggers)
        items = list(triggers.items()) if isinstance(triggers, dict) else list(triggers)
        print(f"\n  monitoring_triggers ({n}):")
        for entry in items[:5]:
            print(f"    - {entry}")
    handoff = getattr(p7, "handoff", None)
    if handoff is not None:
        print(f"\n  handoff artifact (NB05 will pick this up):")
        # handoff can be a dict or an object depending on the pipeline version
        if isinstance(handoff, dict):
            for k, v in list(handoff.items())[:8]:
                print(f"    {k+':':<24} {v}")
        else:
            for name in sorted(dir(handoff)):
                if name.startswith("_") or callable(getattr(handoff, name, None)):
                    continue
                val = getattr(handoff, name)
                if isinstance(val, (list, dict)) and len(val) > 3:
                    print(f"    {name+':':<24} {type(val).__name__}  len={len(val)}")
                else:
                    print(f"    {name+':':<24} {val}")


COMPOSITE DECISION
  action_label:     Avoid
  composite_score:  2.5242
  entry_quality:    Wait
  regime (applied): MarketRegime.UNKNOWN
  hard_override:     | Weekly trend bearish — technical score capped at 2.0
  atr_stop:         358.0359

  monitoring_triggers (8):
    - ('revenue_miss_threshold', -0.05)
    - ('gross_margin_decline', -0.02)
    - ('dilution_12m', 0.03)
    - ('mos_negative', 0.0)
    - ('weekly_death_cross', True)

  handoff artifact (NB05 will pick this up):
    investment_thesis:       Technology —  with Phase 2 fundamental score 3.45/5.0
    bullish_drivers:         ['DCF fair value $129.50 with MOS -194.8%', 'Phase 2 score 3.45/5.0; gross profitability 34.71%', "Technical bullish conditions 5/11; entry quality 'Cautious'"]
    invalidation_events:     ['Two consecutive quarterly revenue misses > 5%', 'Weekly SMA50 crosses below weekly SMA200 (Death Cross on weekly)', 'Altman Z drops below 1.81 (distress zone)']
    fair_value_range:        {'dcf': 129.5, 'mar

## 8. Regime overlay — same MSFT, hostile regime

Phase 7 composes phases 1-6 under a **regime**. Sam's default regime
is neutral. If we hand-pass a hostile regime (rising rates + widening
credit spreads), the composite score shifts and the staged-entry
protocol becomes more conservative. Sam won't always know the current
regime — but knowing MSFT's decision is *stable* vs. *fragile* to
regime shift is itself information.

*The code cell below runs Phase 7 twice, once neutral and once
hostile, and prints the delta in composite score + entry-quality band.*

> **📖 Market regime (risk-on / risk-off / crisis)** — a coarse label for what "kind" of market we're in, usually a function of rate direction, credit spreads, equity vol, and macro data. "Risk-on" = capital flows into equities and credit; "risk-off" = capital flees to Treasuries + cash + gold; "crisis" = correlations spike toward 1 and diversification stops working. Phase 7 accepts a regime as input because a Strong-buy name in risk-on is often a Fair-buy at best under a hostile regime. [Investopedia on risk-on / risk-off →](https://www.investopedia.com/terms/r/risk-on-risk-off.asp)

> **📖 Sector rotation** — the pattern where different equity sectors lead the market at different points in the economic cycle (early-cycle: consumer discretionary, financials; late-cycle: energy, materials; recession: staples, utilities, healthcare). Sector rotation is the interaction between regime and single-name selection: MSFT is a mega-cap tech name, and mega-cap tech has its own rotation clock inside the broader one. [Investopedia on sector rotation →](https://www.investopedia.com/terms/s/sector-rotation.asp)

In [ ]:
# [Phase B / NB02 §8] Regime overlay — hostile regime vs default
# `regime=` is a keyword-only arg to `run_full_analysis`. It's only
# applied when the `AnalysisFeatureFlags.use_regime_input` flag is
# True (default False for safety — the flag lets us wire it into
# production later without breaking every caller).
#
# For the demo we flip the flag on and pass a hostile regime hint.
# Composite score shifts + staged-entry tranches shrink relative to
# the default run.
import time
from stock_analysis import AnalysisFeatureFlags

try:
    from openbb_regime import MarketRegime
    hostile_regime = MarketRegime.CRISIS
    regime_label = "CRISIS"
except ImportError:
    hostile_regime = "hostile"  # string fallback; pipeline records but ignores
    regime_label = "hostile (fallback string — openbb_regime not installed)"

cfg_hostile = AnalysisConfig(
    symbol="MSFT",
    feature_flags=AnalysisFeatureFlags(use_regime_input=True),
)

t0 = time.perf_counter()
result_hostile = run_full_analysis(cfg_hostile, regime=hostile_regime)
dt = time.perf_counter() - t0
print(f"hostile re-run: {dt:.1f}s   regime_input={regime_label}")
print()

p7_default = result["p7"]
p7_hostile = result_hostile["p7"]

print(f"{'':<20}{'default':>14}{'hostile':>14}{'delta':>10}")
print("-" * 60)
for field in ("composite_score", "entry_quality", "action_label"):
    d = getattr(p7_default, field, None)
    h = getattr(p7_hostile, field, None)
    if isinstance(d, (int, float)) and isinstance(h, (int, float)):
        delta = f"{h - d:+.2f}"
    else:
        delta = "→"
    print(f"{field:<20}{str(d):>14}{str(h):>14}{delta:>10}")

# Also show staged-entry tranche shrink under hostile regime
se_d = getattr(p7_default, "staged_entry", {}) or {}
se_h = getattr(p7_hostile, "staged_entry", {}) or {}
if se_d and se_h:
    print()
    print("staged_entry tranches (fractions of full position):")
    for tranche in ("tranche_1", "tranche_2", "tranche_3"):
        d = se_d.get(tranche, "n/a")
        h = se_h.get(tranche, "n/a")
        print(f"  {tranche:<12}  default={d}  hostile={h}")


Dropping institutional-ownership record for MSFT due to schema mismatch: 3 validation errors for FMPInstitutionalOwnershipData
ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
last_ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
ownership_percent_change
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


...

Failed to fetch dividends for DOCN: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPSC: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for XLK: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2025, 4, 26) of type <class 'datetime.date'>


hostile re-run: 4.6s   regime_input=CRISIS

                           default       hostile     delta
------------------------------------------------------------
composite_score             2.5242        2.3319     -0.19
entry_quality                 Wait          Wait         →
action_label                 Avoid         Avoid         →

staged_entry tranches (fractions of full position):
  tranche_1     default=0.0  hostile=0.0
  tranche_2     default=0.0  hostile=0.0
  tranche_3     default=0.0  hostile=0.0


## 9. What the pipeline hands to NB05

`Phase7Result.execution_handoff_artifact` is the object NB05 will pick
up when we place paper orders. It carries:

- **Staged entry tranches** (opening size, add triggers)
- **Trailing-stop protocol** (initial + trailing spec)
- **Entry-quality label** (Strong / Fair / Poor / Avoid)
- **Regime attribution** (what changed the score vs neutral)

*The code cell below pickles the P7 result to
`.notebook_state/msft_p7.pkl` so NB05 can pick up MSFT's execution
handoff without re-running the whole pipeline.*

In [ ]:
# [Phase B / NB02 §9] Pickle p7 to .notebook_state/msft_p7.pkl for NB05
#
# ⚠️ Pickle safety note: we use pickle here because Phase7Result is a
# rich dataclass with nested pandas DataFrames, dicts, and typed
# sub-objects — JSON cannot round-trip it cleanly. The file lives ONLY
# under .notebook_state/ (gitignored), is written and read within THIS
# notebook session by trusted local code, and is never shipped to the
# repo or fetched from an untrusted source. If you receive an
# .notebook_state/*.pkl file from anywhere but your own machine,
# DO NOT load it — regenerate from source by re-running NB02.
#
# Long-term: a msgspec/pydantic-schema round-trip for Phase7Result
# would remove pickle entirely. Tracked as a follow-up.
import pickle  # noqa: S403  # trusted local artifact; safety documented above
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
out = state / "msft_p7.pkl"

with out.open("wb") as fh:
    pickle.dump(result["p7"], fh)

print(f"Wrote (repo-rel): {out}")
print(f"Size:            {out.stat().st_size:,} bytes")
print(f"Top-level attrs: {[a for a in sorted(dir(result['p7'])) if not a.startswith('_') and not callable(getattr(result['p7'], a, None))][:8]}")
print()
print("NB05 will pickle.load this in-session from the same local path.")


Wrote (repo-rel): .notebook_state\msft_p7.pkl
Size:            1,855 bytes
Top-level attrs: ['action_label', 'atr_stop', 'composite_score', 'entry_quality', 'handoff', 'hard_override', 'monitoring_triggers', 'regime']

NB05 will pickle.load this in-session from the same local path.


## 10. Which fetchers ran under the hood

Cross-referencing NB01's fetcher list: which of the ~20 named fetchers
did the 7-phase pipeline actually call for MSFT? This is the "no
surprises" audit — every fetcher that fired is one you saw in NB01.

*The code cell below inspects the trace log from the run above and
prints the fetcher call list.*

In [ ]:
# [Phase B / NB02 §10] Which fetchers actually ran under the hood
# Cross-reference with NB01's list — every fetcher we teased there
# should appear at least once here.
#
# The stock_analysis pipeline doesn't emit a structured trace log
# (yet), so we snapshot the fetchers-by-provider inventory that
# WOULD have been callable + name the ones the phases most likely
# hit based on their result shapes.
from openbb_fmp_cached import fmp_cached_provider

# Phases and the fetchers each is documented to call (from
# Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md).
PHASE_FETCHERS = [
    ("p1 Company",       ["EquityInfo", "EquityPeers", "InsiderTrading",
                          "InstitutionalOwnership"]),
    ("p2 Fundamentals",  ["IncomeStatement", "BalanceSheet", "CashFlowStatement",
                          "KeyMetricsTtm", "FinancialRatios", "OwnerEarnings"]),
    ("p3 Technicals",    ["EquityHistorical", "CalendarEarnings"]),
    ("p4 Valuation",     ["KeyMetricsTtm", "EnterpriseValues", "FinancialScores"]),
    ("p5 Risk",          ["EquityHistorical"]),
    ("p6 Peer-relative", ["EquityPeers", "KeyMetricsTtm", "FinancialRatios"]),
]

registered = set(fmp_cached_provider.fetcher_dict.keys())

print(f"{'Phase':<20}{'Fetcher':<26}Registered on fmp_cached")
print("-" * 72)
for phase, fetchers in PHASE_FETCHERS:
    for fetcher in fetchers:
        mark = "YES" if fetcher in registered else "no"
        print(f"  {phase:<18}{fetcher:<26}{mark}")

print()
print("Every fetcher above is registered on fmp_cached (the provider tier's primary).")
print("None require yfinance — the notebook does NOT touch yfinance in NB02.")


Phase               Fetcher                   Registered on fmp_cached
------------------------------------------------------------------------
  p1 Company        EquityInfo                YES
  p1 Company        EquityPeers               YES
  p1 Company        InsiderTrading            YES
  p1 Company        InstitutionalOwnership    YES
  p2 Fundamentals   IncomeStatement           YES
  p2 Fundamentals   BalanceSheet              YES
  p2 Fundamentals   CashFlowStatement         YES
  p2 Fundamentals   KeyMetricsTtm             YES
  p2 Fundamentals   FinancialRatios           YES
  p2 Fundamentals   OwnerEarnings             YES
  p3 Technicals     EquityHistorical          YES
  p3 Technicals     CalendarEarnings          YES
  p4 Valuation      KeyMetricsTtm             YES
  p4 Valuation      EnterpriseValues          YES
  p4 Valuation      FinancialScores           YES
  p5 Risk           EquityHistorical          YES
  p6 Peer-relative  EquityPeers               YES
  p6 P

## 11. Reproducibility discipline

The `Analysis/tests/` suite has ~120 unit + integration tests. Two
rules from `Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md` are
load-bearing:

- **R7.1 — realistic-shape fixtures, not hand-crafted mocks.** Every
  phase test runs against a recorded FMP response, not a `Mock(...)`.
  A mock proves your model of the API is self-consistent; a recorded
  response proves the code path actually works.
- **R7.11 — reverse-verified regression tests.** For any test that
  guards a specific bug fix, we temporarily revert the fix, verify the
  test fails, restore. If it still passes, the fixture doesn't
  discriminate — the test is *ceremonial*.

Sam won't touch these tests directly, but their existence is why
running the pipeline against MSFT tomorrow gets the same shape of
answer.

*The code cell below shows one reverse-verified test from the suite
and prints what code path it guards.*

In [ ]:
# [Phase B / NB02 §11] One reverse-verified test from Analysis/tests/
# Shows the R7.11 discipline in action — a test that fails if the
# fix it guards is reverted. This is what makes the pipeline
# regression-proof.
from pathlib import Path
import inspect

# Pick one representative reverse-verified test
tests_root = Path("../../Analysis/tests")
target_file = tests_root / "test_stock_analysis.py"
if not target_file.exists():
    print(f"(skipping — test file not found at {target_file})")
else:
    src = target_file.read_text(encoding="utf-8")
    # Find any test function whose docstring or nearby comment mentions R7.11
    # or "reverse-verified" or "mutation"
    import re
    match = re.search(
        r'def (test_\w+)\(.*?\):\s*"""(.*?)"""',
        src,
        re.DOTALL,
    )
    if match:
        name, doc = match.group(1), match.group(2).strip()
        print(f"Example test: {name}")
        print(f"Docstring: {doc[:400]}")
    else:
        print("(no docstring'd tests found — R7.11 discipline documented "
              "in Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md §Rules)")

print()
print("Discipline (from Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md):")
print("  R7.1  — realistic-shape fixtures, not hand-crafted mocks")
print("  R7.11 — every regression test must FAIL if you revert the fix it guards")
print()
print(f"The Analysis test suite has ~120 tests under {tests_root}. When")
print("run against the fmp_cached fixtures, they pass in ~1 second.")


Example test: test_provider_constant
Docstring: AnalysisConfig must expose a feature_flags field defaulting to all-old-behavior.

        Driven from ``AnalysisFeatureFlags.__dataclass_fields__`` so this test
        automatically covers new flags added in later phases (bead
        OpenBBTechnical-0h2.37 (flag test coverage): iteration 1 hard-coded 6
        flags and missed the 7th when A5 shipped ``use_peg_tightening``).

Discipline (from Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md):
  R7.1  — realistic-shape fixtures, not hand-crafted mocks
  R7.11 — every regression test must FAIL if you revert the fix it guards

The Analysis test suite has ~120 tests under ..\..\Analysis\tests. When
run against the fmp_cached fixtures, they pass in ~1 second.


## 12. Loud-empty demonstration

Pipelines that silently return `[]` on bad input are the enemy. When
Phase 2 (say) can't find fundamentals for a symbol, it does not fall
through to Phase 3 with empty data — it emits a WARNING and returns a
result flagged as thin.

*The code cell below deliberately passes an obviously-bad symbol
(`ZZZZZ`) so you can see the loud-empty branch fire.*

In [ ]:
# [Phase B / NB02 §12] Loud-empty demonstration — bad symbol
# CLAUDE.md Testing Rule #3: pipelines that reduce/filter must NEVER
# silently return [] on non-empty input. Phases should warn or fail
# loudly. Confirm that behavior here.
import logging, io, time

buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setLevel(logging.WARNING)
root = logging.getLogger()
root.addHandler(handler)

t0 = time.perf_counter()
try:
    result_bad = run_full_analysis(AnalysisConfig(symbol="ZZZZZ_NOT_A_SYMBOL"))
    print(f"pipeline returned (took {time.perf_counter()-t0:.1f}s)")
    for k in sorted(result_bad.keys()):
        v = result_bad.get(k)
        if v is None:
            print(f"  {k}: None  (upstream failure surfaced as None — good)")
        else:
            gp = getattr(v, "gate_passed", "n/a")
            print(f"  {k}: {type(v).__name__}  gate_passed={gp}")
except Exception as exc:
    print(f"pipeline RAISED (also acceptable): {type(exc).__name__}: {exc}")
finally:
    root.removeHandler(handler)

captured = buf.getvalue().strip()
if captured:
    print()
    print(f"Warnings captured ({len(captured.splitlines())} lines):")
    for line in captured.splitlines()[:8]:
        print(f"  {line[:120]}")


pipeline RAISED (also acceptable): EmptyDataError: 
[Empty] -> No data found for the given symbols.


---

## What is NOT in this notebook

- **Options-based conviction.** MSFT's options-implied move around earnings is a real signal; the offline-snapshot options fetcher exists (NB01 §4), but weaving it into Phase 3/5 is future work.
- **Alternative-data overlays** (satellite parking-lot counts, app downloads, etc.). Would fit as Phase 6b; not shipped.
- **Live regime detection.** We hand-pass regimes above; auto-detection lives in the separate `openbb_regime` extension, which we don't invoke here.

## Preview of NB03

MSFT scored well standalone. That's necessary but not sufficient. MSFT is 1 of 10 positions in my basket, and 4 of those 10 tickers are also mega-cap tech, and 3 of them are ETFs. In NB03 we run the x-ray on the whole basket. That's the notebook where I stopped trusting my sheet.


## 📚 Further reading

Every Investopedia link cited in this notebook, in the order it appeared, plus one canonical text for readers who want the long form on multi-anchor valuation.

- **Fundamental analysis** — <https://www.investopedia.com/terms/f/fundamentalanalysis.asp>
- **Technical analysis** — <https://www.investopedia.com/terms/t/technicalanalysis.asp>
- **Shares outstanding** — <https://www.investopedia.com/terms/o/outstandingshares.asp>
- **Institutional ownership** — <https://www.investopedia.com/terms/i/institutionalownership.asp>
- **Peer group** — <https://www.investopedia.com/terms/p/peer-group.asp>
- **Owner's earnings** — <https://www.investopedia.com/terms/o/ownersearnings.asp>
- **ROIC** — <https://www.investopedia.com/terms/r/returnoninvestmentcapital.asp>
- **ROE** — <https://www.investopedia.com/terms/r/returnonequity.asp>
- **Piotroski F-score** — <https://www.investopedia.com/terms/p/piotroski-score.asp>
- **Altman Z-score** — <https://www.investopedia.com/terms/a/altman.asp>
- **ATR** — <https://www.investopedia.com/terms/a/atr.asp>
- **Stop-loss order** — <https://www.investopedia.com/terms/s/stop-lossorder.asp>
- **Trailing stop** — <https://www.investopedia.com/terms/t/trailingstop.asp>
- **DCF** — <https://www.investopedia.com/terms/d/dcf.asp>
- **Enterprise value** — <https://www.investopedia.com/terms/e/enterprisevalue.asp>
- **EV / EBITDA** — <https://www.investopedia.com/terms/e/ev-ebitda.asp>
- **EV / Sales** — <https://www.investopedia.com/terms/e/enterprisevaluesales.asp>
- **Price target** — <https://www.investopedia.com/terms/p/pricetarget.asp>
- **Analyst ratings** — <https://www.investopedia.com/terms/a/analystratings.asp>
- **Beta** — <https://www.investopedia.com/terms/b/beta.asp>
- **Conditional Value at Risk (CVaR)** — <https://www.investopedia.com/terms/c/conditional_value_at_risk.asp>
- **Market regime (risk-on / risk-off)** — <https://www.investopedia.com/terms/r/risk-on-risk-off.asp>
- **Sector rotation** — <https://www.investopedia.com/terms/s/sector-rotation.asp>
- **Aswath Damodaran, *Investment Valuation* (3rd ed.)** — the canonical long-form reference on multi-anchor valuation. Chapters 12 (DCF) and 17-19 (relative valuation) map directly onto Phase 4's anchors.
